# Notebook 09 — Residue Calculation

**Purpose:** Convert crop production figures to residue quantities using standard Residue-to-Product Ratios (RPR), then split residue into burned and recoverable fractions using state- and crop-specific burning fractions.

**Parameters:**
- RPR: Rice 1.50, Wheat 1.10
- Burn fraction: Rice/Punjab 80%, Rice/Haryana 65%, Wheat 50%
- Recovery efficiency: 70% of un-burned residue

**Input:** `Data/Processed/crop_production_clean.csv`  
**Output:** `Data/Processed/residue_calc.csv`

In [1]:
import pandas as pd

# Load cleaned crop data 
crop = pd.read_csv('../Data/Processed/crop_production_clean.csv')
print(f"Shape: {crop.shape}")
print(crop.head())

Shape: (611, 5)
    state  district         year  crop  production_t
0  Punjab  Amritsar  2010 - 2011  Rice      503000.0
1  Punjab  Amritsar  2011 - 2012  Rice      482000.0
2  Punjab  Amritsar  2012 - 2013  Rice      544000.0
3  Punjab  Amritsar  2013 - 2014  Rice      518000.0
4  Punjab  Amritsar  2014 - 2015  Rice      505000.0


In [2]:
# Clean state column so state-based logic works correctly 
# The XLS may encode state names as '1. Punjab' — strip the prefix.
crop['state']    = crop['state'].str.replace(r'^\d+\.\s*', '', regex=True).str.strip()
crop['district'] = crop['district'].str.replace(r'^\d+\.\s*', '', regex=True).str.strip()

print("Unique states after cleaning:", crop['state'].unique())


Unique states after cleaning: ['Punjab' 'Haryana']


In [3]:
# Residue-to-Product Ratio (RPR) 
# Converts crop production (tonnes grain) → residue / straw (tonnes).
# Source: MNRE / ICAR standard values.
RPR = {'Rice': 1.50, 'Wheat': 1.10}

crop['RPR']                 = crop['crop'].map(RPR)
crop['residue_generated_t'] = crop['production_t'] * crop['RPR']

print(crop[['crop', 'production_t', 'RPR', 'residue_generated_t']].head())


   crop  production_t  RPR  residue_generated_t
0  Rice      503000.0  1.5             754500.0
1  Rice      482000.0  1.5             723000.0
2  Rice      544000.0  1.5             816000.0
3  Rice      518000.0  1.5             777000.0
4  Rice      505000.0  1.5             757500.0


In [4]:
# Burning fraction 
# Fraction of generated residue that is burned in the field.
# Punjab rice: 80% (higher mechanisation, faster turnaround pressure)
# Haryana rice: 65% (slightly lower burning prevalence)
# Wheat (both states): 50%
# Source: IARI / APCAB estimates used in literature.
def get_burn_fraction(row):
    if row['crop'] == 'Rice':
        return 0.80 if row['state'] == 'Punjab' else 0.65
    else:   # Wheat
        return 0.50

crop['burn_fraction'] = crop.apply(get_burn_fraction, axis=1)

# Verify the mapping is applied correctly
print("Burn fraction by state + crop:")
print(
    crop[['state', 'crop', 'burn_fraction']]
    .drop_duplicates()
    .sort_values(['state', 'crop'])
    .to_string(index=False)
)


Burn fraction by state + crop:
  state  crop  burn_fraction
Haryana  Rice           0.65
Haryana Wheat           0.50
 Punjab  Rice           0.80
 Punjab Wheat           0.50


In [5]:
# Burned and recoverable residue 
# burned_t       : residue set on fire (pollution source)
# recoverable_t  : residue available for bioenergy (un-burned × 70% efficiency)
#   The 70% efficiency factor accounts for collection, transport, and
#   storage losses in real-world biomass supply chains.
crop['burned_t']       = crop['residue_generated_t'] * crop['burn_fraction']
crop['recoverable_t']  = crop['residue_generated_t'] * (1 - crop['burn_fraction']) * 0.70

print(crop[['state', 'district', 'year', 'crop',
            'residue_generated_t', 'burned_t', 'recoverable_t']].head(8))


    state  district         year  crop  residue_generated_t  burned_t  \
0  Punjab  Amritsar  2010 - 2011  Rice             754500.0  603600.0   
1  Punjab  Amritsar  2011 - 2012  Rice             723000.0  578400.0   
2  Punjab  Amritsar  2012 - 2013  Rice             816000.0  652800.0   
3  Punjab  Amritsar  2013 - 2014  Rice             777000.0  621600.0   
4  Punjab  Amritsar  2014 - 2015  Rice             757500.0  606000.0   
5  Punjab  Amritsar  2015 - 2016  Rice             769500.0  615600.0   
6  Punjab  Amritsar  2016 - 2017  Rice             861000.0  688800.0   
7  Punjab  Amritsar  2017 - 2018  Rice            1015500.0  812400.0   

   recoverable_t  
0       105630.0  
1       101220.0  
2       114240.0  
3       108780.0  
4       106050.0  
5       107730.0  
6       120540.0  
7       142170.0  


In [6]:
# Summary statistics 
print(crop[['residue_generated_t', 'burned_t', 'recoverable_t']].describe().round(0))


       residue_generated_t   burned_t  recoverable_t
count                611.0      611.0          611.0
mean              822611.0   531228.0       203968.0
std               449354.0   332692.0       141972.0
min                 7500.0     4875.0         1837.0
25%               462350.0   294600.0        95480.0
50%               805200.0   489500.0       160860.0
75%              1092750.0   690158.0       311658.0
max              2296500.0  1837200.0       758065.0


In [8]:
# Save 
crop.to_csv('../Data/Processed/residue_calc.csv', index=False)
print("Saved: Data/Processed/residue_calc.csv")
print(f"   Rows    : {len(crop)}")
print(f"   Columns : {crop.columns.tolist()}")

✅ Saved: Data/Processed/residue_calc.csv
   Rows    : 611
   Columns : ['state', 'district', 'year', 'crop', 'production_t', 'RPR', 'residue_generated_t', 'burn_fraction', 'burned_t', 'recoverable_t']
